# OM2 input trends: tas & huss (OAFlux)

_Split from `trend_obs_om2-inputs.ipynb` (tas/huss). Shared helpers live in `functions.py`._


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
# import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import scipy
# import re
# import seaborn as sns
# import pandas as pd
import matplotlib.gridspec as gridspec
# from scipy.stats import norm
# from mpl_toolkits.basemap  import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
import xesmf as xe
from matplotlib.gridspec import GridSpec
import xesmf as xe
import re
import pandas as pd
import matplotlib
import os, glob


from functools import partial

%matplotlib inline


In [ ]:
import sys, os
# make this repo's functions.py importable from any subfolder
_p = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_p, 'functions.py')):
        sys.path.insert(0, _p); break
    _p = os.path.dirname(_p)
from functions import *  # shared helper functions


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
# import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import scipy
# import re
# import seaborn as sns
# import pandas as pd
import matplotlib.gridspec as gridspec
# from scipy.stats import norm
# from mpl_toolkits.basemap  import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
import xesmf as xe
from matplotlib.gridspec import GridSpec
import xesmf as xe
import re
import pandas as pd
import matplotlib
import os, glob


from functools import partial

%matplotlib inline


# Find trend files for uas, vas, tas and huss 

uas and vas from - 

tas and huss from oaflux

In [ ]:
    
years = np.arange(1982, 2018, 1)

tas_fps = [
    f"/Users/isabelaconde/Desktop/data/oaflux_WHOI/ta_oaflux_{y}.nc"
    for y in years
]

def fix_time(ds, prefix):
    src = ds.encoding.get("source", "")
    year = int(re.search(rf"{re.escape(prefix)}_oaflux_(\d{{4}})\.nc$", src).group(1))

    months = [int(m) for m in ds["time"].values]  # 1..12
    new_time = pd.to_datetime([f"{year}-{m:02d}-15" for m in months])

    return ds.assign_coords(time=("time", new_time))

prefix = "ta"

tas = xr.open_mfdataset(
    tas_fps,
    preprocess=partial(fix_time, prefix=prefix),
    combine="by_coords"
).sortby("time")

tas

In [ ]:

huss_fps = [
    f"/Users/isabelaconde/Desktop/data/oaflux_WHOI/qa_oaflux_{y}.nc"
    for y in years
]

huss = xr.open_mfdataset(
    huss_fps,
    preprocess=partial(fix_time, prefix="qa"),
    combine="by_coords"
).sortby("time")


In [ ]:
xr.open_dataset('/Users/isabelaconde/Desktop/data/oaflux_WHOI/qa_oaflux_1982.nc')

In [ ]:
tas_trend = find_trend_values(pacific_mask(tas.tmp2m)*tas.tmp2m).sel(lat = slice(-20,20), lon = slice(120,290))
huss_trend = find_trend_values(pacific_mask(huss.hum2m)*huss.hum2m).sel(lat = slice(-20,20), lon = slice(120,290))

In [ ]:
# add attributies from tas adn trend
tas_trend = tas_trend.assign_attrs(tas.attrs)
tas_trend.attrs["comments"] = "Trend per month over 1982-2017 (monthly-mean 2 m air temperature)."

huss_trend = huss_trend.assign_attrs(huss.attrs)
huss_trend.attrs["comments"] = "Trend per month over 1982-2017 (monthly-mean 2 m specific humidity)."

#save to oaflux_WHOI directory
tas_trend.to_netcdf("/Users/isabelaconde/Desktop/data/oaflux_WHOI/tas_oaflux_trend_1982-2017.nc")
huss_trend.to_netcdf("/Users/isabelaconde/Desktop/data/oaflux_WHOI/huss_oaflux_trend_1982-2017.nc")